In [12]:
import os
import sys
import yaml
import pandas as pd
import seaborn as sns
current_dir = os.path.dirname(os.path.abspath('.'))
project_root = os.path.abspath(os.path.join(current_dir, "../.."))
sys.path.insert(0, project_root)
from utils.plots import Pearson_correlation, Bar_plot, plot_numerical_data
import matplotlib.pyplot as plt
plt.style.use('ggplot')

In [13]:
from utils.utils import to_jsonl
from functions.make_dataset import save_data
from functions.model_selection import grid_search_single_model_StratifiedKFold, randomized_single_model_grid_search
from functions.train_model import train_model, save_model
from functions.evaluate_model import evaluate_reg_model, MetricsOrchestrator
from functions.predict_model import make_prediction_reg
from functions.cross_validate import cross_validate_kfold
from functions.single_model_reg import SingleModelOrchestrator

In [14]:
with open(os.path.join("../config/config.yaml"), "r") as f:
    config = yaml.safe_load(f)
        
with open(os.path.join( "../config/pipeline.yaml"), "r") as f:
    config_pipe = yaml.safe_load(f)  
    
with open(os.path.join( "../config/pipeline.yaml"), "r") as f:
    config_model = yaml.safe_load(f)

In [15]:
pipeline_name='pipeline1'

In [16]:
# Datasets X_train
X_train = pd.read_parquet(
    os.path.join(
        config['init_path'],
        config['data']['feature_eng'],
        f"X_train_feat_eng_{pipeline_name}.parquet")
)
   
y_train = pd.read_parquet(
    os.path.join(
        config['init_path'],
        config['data']['feature_eng'],
        f"y_train_feat_eng_{pipeline_name}.parquet")
)

# Datasets Y_train
X_val = pd.read_parquet(
    os.path.join(
        config['init_path'],
        config['data']['feature_eng'],
        f"X_val_feat_eng_{pipeline_name}.parquet")
)

y_val = pd.read_parquet(
    os.path.join(
        config['init_path'],
        config['data']['feature_eng'],
        f"y_val_feat_eng_{pipeline_name}.parquet")
)

In [17]:
model_name="RandomForestRegressor"

In [18]:
model_orchestrator = SingleModelOrchestrator()
model_config = model_orchestrator.apply(model_name)  

scoring="neg_mean_absolute_percentage_error"

In [19]:
best_paramns = randomized_single_model_grid_search(
            X_train, 
            y_train, 
            model_config['model'], 
            model_config['param_distributions'], 
            scoring=scoring
            ) 

In [20]:
model_reg = train_model(
    X_train, 
    y_train, 
    model_config['model'], 
    best_paramns)

In [21]:
df_cv = cross_validate_kfold(
        X_train=X_train, 
        y_train=y_train, 
        model=model_reg,
        model_config=model_config,
        score=scoring        
        )

In [22]:
df_cv

,fold,train_score,val_score,scoring,model_type,model,timestamp
0,0,-0.0046,-0.0091,neg_mean_absolute_percentage_error,single_model,RandomForestRegressor,2026-05-13T13:48:50.181641
1,1,-0.0046,-0.0088,neg_mean_absolute_percentage_error,single_model,RandomForestRegressor,2026-05-13T13:48:50.181641
2,2,-0.0047,-0.0080,neg_mean_absolute_percentage_error,single_model,RandomForestRegressor,2026-05-13T13:48:50.181641
3,3,-0.0046,-0.0102,neg_mean_absolute_percentage_error,single_model,RandomForestRegressor,2026-05-13T13:48:50.181641
4,4,-0.0047,-0.0088,neg_mean_absolute_percentage_error,single_model,RandomForestRegressor,2026-05-13T13:48:50.181641


In [23]:
print(f"Mean train score {df_cv['score'].unique()[0]}: {df_cv['train_score'].mean()} +- {df_cv['train_score'].std()}", end='\n')

KeyError: 'score'